# 02. 임베딩 → FAISS → Qdrant → 필터 검색 (실습 1 · Step 3~6)

**이 노트북의 목표:** 01에서 만든 청크 2,197개를 **숫자 벡터로 바꾸고(임베딩)**, 두 가지 방식으로 "의미가 비슷한 청크"를 찾아본다.

```
outputs/chunks.jsonl
   → Step 3. BGE-M3 임베딩  → outputs/embeddings.npy (캐시)
   → Step 4. FAISS HNSW      : 내 컴퓨터 메모리 안에서 빠르게 검색 (실험용)
   → Step 5. Qdrant 업로드   : 벡터 + 꼬리표(payload)를 DB 서버에 저장
   → Step 6. 필터 검색       : "반도체 문서만", "2026년 자료만" 같은 조건 + 의미 검색
```

| 도구 | 한 줄 설명 | 이 프로젝트에서 역할 |
|---|---|---|
| **BGE-M3** | 문장을 1,024개 숫자로 바꿔주는 다국어 임베딩 모델 | 한국어 보고서 → 벡터 |
| **FAISS** | Meta가 만든 벡터 검색 라이브러리 (파일·메모리 기반) | 빠른 실험, 정답(Exact)과 비교 |
| **Qdrant** | 벡터 전용 DB 서버 (Docker로 실행) | 메타데이터 필터 + 실제 서비스용 저장소 |

## Step 3. 임베딩 (Embedding)

컴퓨터는 글자의 "의미"를 직접 비교하지 못한다. 그래서 문장을 **숫자 목록(벡터)**으로 바꾼 뒤, 벡터끼리 **얼마나 같은 방향을 가리키는지(코사인 유사도)**로 의미가 가까운지 판단한다.

- `"HBM 메모리"` 와 `"고대역폭 메모리"` → 글자는 달라도 벡터가 가까움 ✅
- `"HBM 메모리"` 와 `"영국 AI 투자"` → 벡터가 멂

임베딩은 청크 2,197개를 모델에 통과시켜야 해서 몇 분 걸린다. 그래서 **한 번 계산한 결과를 `outputs/embeddings.npy` 파일로 저장(캐시)**해 두고, 다음부터는 파일만 읽는다.

In [1]:
# ─────────────────────────────────────────────────────────────
# Step 3-1. 라이브러리 불러오기 + 01에서 만든 청크 읽기
# ─────────────────────────────────────────────────────────────
import json
import time
from pathlib import Path

import numpy as np    # 벡터(숫자 배열)를 다루는 기본 도구
import pandas as pd   # 검색 결과를 표로 보기 위해 사용

PROJECT_ROOT = Path("..").resolve()           # notebooks/ 의 한 칸 위 = 프로젝트 루트
OUT_DIR = PROJECT_ROOT / "outputs"
CHUNKS_PATH = OUT_DIR / "chunks.jsonl"        # 01 노트북의 결과물
EMB_PATH = OUT_DIR / "embeddings.npy"         # 이 노트북에서 만들 임베딩 캐시

# jsonl = 한 줄에 청크 하나 → 줄마다 json.loads 로 읽는다
with open(CHUNKS_PATH, encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

# ★ 01에서 강조한 "공통 열쇠" 확인: chunks[i]의 chunk_id 가 정확히 i 여야 한다.
#   이게 어긋나면 embeddings[i] 가 엉뚱한 청크의 벡터가 되어버린다.
assert all(c["chunk_id"] == i for i, c in enumerate(chunks)), "chunk_id 순서가 어긋남! 01 노트북을 다시 실행하세요."

print(f"✅ 청크 로딩 완료: {len(chunks)}개 (참고값: 2,197)")
print("   첫 청크의 꼬리표:", {k: v for k, v in chunks[0].items() if k != "text"})

✅ 청크 로딩 완료: 2197개 (참고값: 2,197)
   첫 청크의 꼬리표: {'page': 2, 'source': '1. SPRi AI Brief_10월호_산업동향_202510', 'year': 2025, 'publisher': 'SPRi', 'category': 'AI동향', 'chunk_id': 0}


In [2]:
# ─────────────────────────────────────────────────────────────
# Step 3-2. BGE-M3 모델 불러오기
#
# - 처음 실행하면 Hugging Face에서 모델(약 2.3GB)을 내려받는다. (한 번만)
# - devices: 계산을 어디서 할지
#     "mps"  = Apple Silicon(M1~M5) GPU  → CPU보다 몇 배 빠름
#     "cuda" = NVIDIA GPU
#     "cpu"  = 둘 다 없을 때
# - use_fp16: 숫자를 절반 정밀도(16비트)로 계산 → 속도↑ 메모리↓, 검색 품질 차이는 거의 없음
#             (CPU에서는 fp16이 오히려 느리거나 안 돼서 끈다)
# ─────────────────────────────────────────────────────────────
import logging
import torch

# macOS에서 FlagEmbedding을 불러오면 torch가 "Redirects 미지원" 경고를 띄운다 → 실습과 무관해서 숨김
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
from FlagEmbedding import BGEM3FlagModel

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

MODEL_NAME = "BAAI/bge-m3"

model = BGEM3FlagModel(
    MODEL_NAME,
    devices=DEVICE,
    use_fp16=(DEVICE != "cpu"),
    normalize_embeddings=True,   # 벡터 길이를 1로 맞춤 → "내적 = 코사인 유사도"가 되어 계산이 단순해진다
)
print(f"✅ BGE-M3 모델 준비 완료 (device={DEVICE})")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ BGE-M3 모델 준비 완료 (device=mps)


In [3]:
# ─────────────────────────────────────────────────────────────
# Step 3-3. 청크 전체 임베딩 (캐시가 있으면 건너뛰기)
#
# 캐시를 그냥 믿으면 위험한 경우가 있다:
#   01에서 청크 설정(크기·오버랩)을 바꿨는데 예전 embeddings.npy가 남아 있으면
#   → 개수가 달라지므로, "행 개수 == 청크 개수"일 때만 캐시를 쓴다.
#   (설정을 바꿨는데 우연히 개수가 같을 수도 있으니, 그럴 땐 npy 파일을 지우고 다시 실행)
# ─────────────────────────────────────────────────────────────
texts = [c["text"] for c in chunks]

if EMB_PATH.exists() and np.load(EMB_PATH, mmap_mode="r").shape[0] == len(chunks):
    embeddings = np.load(EMB_PATH)
    print(f"📦 캐시 사용: {EMB_PATH.name} 불러옴 (임베딩 계산 생략)")
else:
    print(f"🧮 임베딩 계산 시작: {len(texts)}개 청크 (몇 분 걸림)")
    start = time.time()
    output = model.encode(
        texts,
        batch_size=32,      # 한 번에 32개씩 모델에 넣는다 (메모리 부족하면 16, 8로 줄이기)
        max_length=512,     # 청크가 500자라 512 토큰이면 충분 (BGE-M3 최대는 8192)
    )
    # encode 결과는 dict: dense_vecs(우리가 쓸 것), lexical_weights, colbert_vecs(실습 2에서 다룰 수 있음)
    embeddings = output["dense_vecs"].astype(np.float32)   # FAISS·Qdrant 모두 float32를 쓴다
    np.save(EMB_PATH, embeddings)
    print(f"💾 저장 완료: {EMB_PATH} ({time.time() - start:.0f}초 소요)")

print(f"✅ 임베딩 준비 완료: shape = {embeddings.shape}  ← (청크 수, 벡터 차원 1024)")

📦 캐시 사용: embeddings.npy 불러옴 (임베딩 계산 생략)
✅ 임베딩 준비 완료: shape = (2197, 1024)  ← (청크 수, 벡터 차원 1024)


In [4]:
# ─────────────────────────────────────────────────────────────
# Step 3-4. (눈으로 확인) 임베딩이 정말 "의미"를 담고 있을까?
#
# ① 벡터 길이(norm)가 모두 1인지 → normalize_embeddings=True 가 잘 적용됐는지
# ② 문장 4개를 임베딩해서 서로 얼마나 비슷한지(코사인 유사도) 표로 본다
#    - 정규화된 벡터끼리의 내적(@) = 코사인 유사도 (1에 가까울수록 비슷)
# ─────────────────────────────────────────────────────────────
norms = np.linalg.norm(embeddings, axis=1)
print(f"① 벡터 길이: 최소 {norms.min():.4f} / 최대 {norms.max():.4f}  (모두 1.0 근처면 OK)\n")


def embed_query(text: str) -> np.ndarray:
    """질문(문장 1개)을 1024차원 벡터로 바꾼다. 청크 임베딩과 같은 모델·같은 방식이어야 비교가 된다."""
    vec = model.encode([text], max_length=512)["dense_vecs"][0]
    return vec.astype(np.float32)


demo = [
    "HBM은 고대역폭 메모리다",
    "High Bandwidth Memory는 D램을 수직으로 쌓은 반도체다",   # 같은 뜻, 다른 표현 + 영어 섞임
    "엔비디아 GPU에는 메모리가 필요하다",                     # 관련은 있지만 다른 얘기
    "영국 정부가 AI 산업 보고서를 발간했다",                  # 전혀 다른 주제
]
demo_vecs = np.stack([embed_query(t) for t in demo])
sim = demo_vecs @ demo_vecs.T

print("② 문장끼리 코사인 유사도")
pd.DataFrame(sim, index=[t[:18] for t in demo], columns=[f"문장{i + 1}" for i in range(len(demo))]).round(3)

① 벡터 길이: 최소 0.9997 / 최대 1.0003  (모두 1.0 근처면 OK)



② 문장끼리 코사인 유사도


,문장1,문장2,문장3,문장4
HBM은 고대역폭 메모리다,1.000,0.725,0.550,0.317
High Bandwidth Mem,0.725,1.000,0.516,0.337
엔비디아 GPU에는 메모리가 필요,0.550,0.516,1.000,0.350
영국 정부가 AI 산업 보고서를,0.317,0.337,0.350,0.999


### 📝 Step 3 관찰 메모
- 청크 2,197개 임베딩은 Apple M5(MPS, fp16) 기준 **약 26초**. 두 번째 실행부터는 `embeddings.npy` 캐시를 읽어서 바로 끝난다.
- 같은 뜻을 한국어/영어로 다르게 쓴 문장1·2는 **0.73**, 관련만 있는 문장3은 **0.52~0.55**, 무관한 문장4는 **0.32~0.35**. 의미 순서대로 점수가 나온다.
- 다만 **무관한 문장도 0이 아니라 0.3 정도**가 나온다. 점수의 절대값보다는 **순위**로 판단해야 한다(실습 2에서 "점수 몇 이상이면 정답" 같은 기준을 잡기 어려운 이유).
- 문장4의 자기 자신 유사도가 1.000이 아니라 0.999인 건 fp16(절반 정밀도) 반올림 오차라 무시해도 된다.

## Step 4. FAISS로 검색해보기 (HNSW 인덱스)

질문 벡터와 **청크 2,197개 전부**를 하나씩 비교하면(= Flat, 완전 탐색) 정확하지만, 데이터가 수백만 개가 되면 느려진다.

**HNSW**(Hierarchical Navigable Small World)는 벡터들을 **"친구 관계 그래프"**로 미리 연결해 두고, 가까운 친구를 따라가며 정답 근처로 빠르게 이동하는 방식이다. 전부 비교하지 않으니 빠르지만, 가끔 진짜 1등을 놓칠 수 있다(= 근사 검색, ANN).

| 파라미터 | 의미 | 키우면 |
|---|---|---|
| `M` | 벡터 하나가 연결할 친구 수 | 정확도↑ 메모리↑ |
| `efConstruction` | 그래프 만들 때 후보를 몇 개까지 살펴볼지 | 그래프 품질↑ 만드는 시간↑ |
| `efSearch` | 검색할 때 후보를 몇 개까지 살펴볼지 | 정확도↑ 검색 시간↑ |

> Qdrant도 내부적으로 HNSW를 쓴다. 그래서 FAISS로 먼저 감을 잡아두면 Step 5가 쉽다.

In [5]:
# ─────────────────────────────────────────────────────────────
# Step 4-1. FAISS 인덱스 2개 만들기
#
#   ① IndexFlatIP  : 전부 비교하는 "정답지" (Exact). IP = Inner Product(내적)
#   ② IndexHNSWFlat: 그래프로 빠르게 찾는 근사 검색 (ANN)
#
# 벡터를 정규화해 뒀으므로 내적(IP) = 코사인 유사도.
# HNSW 기본 거리는 L2(유클리드 거리)라서, 내적을 쓰도록 METRIC_INNER_PRODUCT를 꼭 지정한다.
# ─────────────────────────────────────────────────────────────
import faiss

# ⚠️ macOS 전용 주의사항: torch(BGE-M3)와 faiss가 각자 OpenMP 라이브러리(libomp)를 들고 와서
#    둘 다 멀티스레드로 돌면 충돌해 커널이 죽는다(Kernel died). faiss를 1스레드로 고정하면 해결된다.
#    청크 2천 개 규모에서는 1스레드여도 순식간이라 속도 차이는 거의 없다.
faiss.omp_set_num_threads(1)

DIM = embeddings.shape[1]    # 1024

# ① 정답지: 모든 벡터와 비교
flat_index = faiss.IndexFlatIP(DIM)
flat_index.add(embeddings)

# ② HNSW: 친구 32명씩 연결
HNSW_M = 32
hnsw_index = faiss.IndexHNSWFlat(DIM, HNSW_M, faiss.METRIC_INNER_PRODUCT)
hnsw_index.hnsw.efConstruction = 200    # 그래프를 꼼꼼하게 만든다 (데이터가 작아서 금방 끝남)

start = time.time()
hnsw_index.add(embeddings)              # 추가하는 순서 = FAISS 내부 번호 0,1,2... = chunk_id
print(f"⏱️ HNSW 그래프 생성: {time.time() - start:.2f}초")

hnsw_index.hnsw.efSearch = 64           # 검색할 때 후보 64개까지 살펴봄 (top-k보다 넉넉하게)

print(f"✅ FAISS 인덱스 준비 완료: Flat {flat_index.ntotal}개 / HNSW {hnsw_index.ntotal}개 (청크 수와 같아야 함)")

⏱️ HNSW 그래프 생성: 0.23초
✅ FAISS 인덱스 준비 완료: Flat 2197개 / HNSW 2197개 (청크 수와 같아야 함)


In [6]:
# ─────────────────────────────────────────────────────────────
# Step 4-2. 검색 함수 + 결과를 표로 보여주는 함수
#
# FAISS는 "몇 번째 벡터인지(번호)"와 "점수"만 돌려준다.
# 번호 = chunk_id 이므로 chunks[번호] 로 원문과 꼬리표를 바로 찾을 수 있다.
# ─────────────────────────────────────────────────────────────
def faiss_search(query: str, k: int = 5, index=hnsw_index) -> list[tuple[int, float]]:
    """질문과 가장 비슷한 청크 k개의 (chunk_id, 점수) 목록을 돌려준다."""
    q = embed_query(query).reshape(1, -1)    # FAISS는 질문도 2차원 배열(질문 수 × 차원)로 받는다
    scores, ids = index.search(q, k)         # 결과도 2차원: [질문0의 결과들]
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]


def show_results(results: list[tuple[int, float]], preview: int = 60) -> pd.DataFrame:
    """(chunk_id, 점수) 목록 → 사람이 읽기 좋은 표"""
    rows = []
    for rank, (cid, score) in enumerate(results, start=1):
        c = chunks[cid]
        rows.append({
            "순위": rank,
            "점수": round(score, 3),
            "발행처": c["publisher"],
            "분류": c["category"],
            "연도": c["year"],
            "쪽": c["page"],
            "chunk_id": cid,
            "내용 미리보기": c["text"][:preview].replace("\n", " "),
        })
    return pd.DataFrame(rows).set_index("순위")


print("✅ 검색 함수 준비 완료: faiss_search(), show_results()")

✅ 검색 함수 준비 완료: faiss_search(), show_results()


In [7]:
# ─────────────────────────────────────────────────────────────
# Step 4-3. 첫 검색: "HBM이란 무엇인가?"
#   01에서 'HBM'이 들어간 청크는 69개였고, 대부분 반도체 문서(반도체특별위원회·KDB·삼일PwC)였다.
# ─────────────────────────────────────────────────────────────
query = "HBM이란 무엇인가?"
print(f"🔎 질문: {query}")
show_results(faiss_search(query, k=5))

🔎 질문: HBM이란 무엇인가?


,점수,발행처,분류,연도,쪽,chunk_id,내용 미리보기
순위,,,,,,,
1,0.585,KDB미래전략연구소,반도체,2024,12,2168,- HBM 기술을 선도16)하던 SK하이닉스가 HBM3(4세대)를 엔비디아 앞 독점...
2,0.578,삼일PwC,반도체,2025,64,1754,DRAM 대비 HBM 시장 규모 및 침투율 1) 침투율: 전체 DRAM 중 HBM ...
3,0.573,KDB미래전략연구소,반도체,2024,11,2165,"2024. 7 제824호 13 2. 메모리 : HBM, PIM ❑ AI 연산에 있..."
4,0.573,KDB미래전략연구소,반도체,2024,11,2166,- PIM 기술은 프로세서와 메모리 영역 간의 거리를 최대한 축소 또는 프로세서와 ...
5,0.557,KDB미래전략연구소,반도체,2024,12,2167,〈표 4〉 HBM 및 GDDR 성능 비교 구분...


In [8]:
# ─────────────────────────────────────────────────────────────
# Step 4-4. 다른 질문도 몇 개 던져보기
#   - 반도체 질문, AI 정책 질문, 최신 뉴스 질문을 섞어서 어떤 문서가 나오는지 본다.
# ─────────────────────────────────────────────────────────────
for q in ["AI 반도체 국가 경쟁력을 높이기 위한 정책", "2026년 반도체 시장 전망", "AI 에이전트 관련 기업 동향"]:
    print(f"\n🔎 질문: {q}")
    print(show_results(faiss_search(q, k=3), preview=45).to_string())


🔎 질문: AI 반도체 국가 경쟁력을 높이기 위한 정책
       점수       발행처   분류    연도   쪽  chunk_id                                        내용 미리보기
순위                                                                                         
1   0.709  반도체특별위원회  반도체  2025  52      1965  제2절 K-AI반도체 강국도약 목표 목표 1. AI 산업 생태계 구축 그림 12 
2   0.708  반도체특별위원회  반도체  2025  45      1947  경제와 안보를 좌우하는 핵심 전략 산업으로 명확히 규정하고, 세제･인프라･R&D･
3   0.707  반도체특별위원회  반도체  2025  12      1850  전망되면서, AI 반도체 수급과 전력 인프라는 국가 차원의 정책 과제로 직결되고 

🔎 질문: 2026년 반도체 시장 전망
       점수         발행처   분류    연도   쪽  chunk_id                                        내용 미리보기
순위                                                                                           
1   0.669       삼일PwC  반도체  2025   7      1633  글로벌 반도체 시장은 데이터센터, 인공지능, 자율주행차, 스마트폰 등 신기술 트렌
2   0.645    반도체특별위원회  반도체  2025  11      1845  제1장 AI반도체의 중요성과 개발동향 제3절 AI반도체의 시장전망 그림 3 | A
3   0.635  KDB미래전략연구소  반도체  2024   4      2149  2. 시장 현황 및 전망 ❑ 글로벌 AI 반도체 시장은 ‘23년 537억달러 규모

🔎 질문: AI 에이전트 

In [9]:
# ─────────────────────────────────────────────────────────────
# Step 4-5. HNSW는 얼마나 정확할까? (Recall@10 측정)
#
# 같은 질문을 ① Flat(정답지)과 ② HNSW로 검색해서,
# 정답 top-10 중 몇 개를 HNSW도 찾아냈는지 비율을 잰다.
#   Recall@10 = |HNSW top10 ∩ Flat top10| / 10
# efSearch를 바꿔가며 "정확도 vs 속도" 균형을 직접 확인한다.
# ─────────────────────────────────────────────────────────────
test_queries = [
    "HBM이란 무엇인가?", "AI 반도체 국가 경쟁력을 높이기 위한 정책", "2026년 반도체 시장 전망",
    "AI 에이전트 관련 기업 동향", "인공지능 인재 양성 방안", "NPU와 GPU의 차이",
    "생성형 AI 저작권 문제", "데이터센터 전력 소비", "유럽연합 AI 법", "파운드리 시장 점유율",
]
q_vecs = np.stack([embed_query(q) for q in test_queries])   # 질문 10개를 한 번에 (10 × 1024)
K = 10

_, exact_ids = flat_index.search(q_vecs, K)                  # 정답지

rows = []
for ef in [8, 16, 32, 64, 128]:
    hnsw_index.hnsw.efSearch = ef
    start = time.perf_counter()
    _, approx_ids = hnsw_index.search(q_vecs, K)
    ms = (time.perf_counter() - start) * 1000 / len(test_queries)
    recall = np.mean([len(set(a) & set(e)) / K for a, e in zip(approx_ids, exact_ids)])
    rows.append({"efSearch": ef, "Recall@10": round(recall, 3), "질문당 검색시간(ms)": round(ms, 3)})

hnsw_index.hnsw.efSearch = 64    # 실험 끝 → 원래 값으로 되돌림
pd.DataFrame(rows).set_index("efSearch")

,Recall@10,질문당 검색시간(ms)
efSearch,,
8,0.91,0.021
16,0.96,0.025
32,0.99,0.033
64,1.00,0.051
128,1.00,0.086


### 📝 Step 4 관찰 메모
- "HBM이란?" 질문의 top-5는 **모두 반도체 문서**(KDB 4개, 삼일PwC 1개)였다. HBM처럼 반도체 문서에 몰려 있는 단어는 Dense만으로도 잘 찾는다.
- 띄어쓰기가 사라진 삼일PwC 청크도 "2026년 반도체 시장 전망"에서 1위로 나왔다. BGE-M3는 단어를 더 작은 조각(subword)으로 쪼개서 보기 때문에 띄어쓰기 영향을 덜 받는다. **띄어쓰기로 단어를 자르는 BM25(실습 2)에서는 결과가 달라질 수 있다.**
- HNSW Recall@10: efSearch=8 → **0.91**, 32 → **0.99**, 64 → **1.00**. 데이터가 2천 개뿐이라 efSearch=64여도 질문당 0.06ms 정도라서, 이 프로젝트에서는 정확도를 우선해 64를 쓴다.
- ⚠️ macOS에서 torch와 faiss를 같이 쓰면 OpenMP 충돌로 **커널이 죽는다**. `faiss.omp_set_num_threads(1)`로 해결했다(Step 4-1 주석 참고).

## Step 5. Qdrant에 업로드

FAISS는 **번호와 점수만** 기억한다. "반도체 문서 중에서만 찾아줘" 같은 조건을 걸려면 결과를 받은 뒤 파이썬으로 걸러야 하는데, 그러면 top-5를 뽑았는데 조건에 맞는 게 1개뿐인 상황이 생긴다.

**Qdrant**는 벡터마다 **payload(꼬리표 묶음)**를 함께 저장하고, **검색하는 도중에** 조건을 적용한다. 그래서 조건에 맞는 것 중에서 정확히 top-k를 돌려준다.

| Qdrant 용어 | 뜻 | 이 프로젝트에서 |
|---|---|---|
| Collection | 테이블 같은 저장 공간 | `policy_reports` |
| Point | 저장 단위 1개 = id + vector + payload | 청크 1개 |
| id | 포인트 번호 | `chunk_id` (FAISS 번호와 동일) |
| payload | 꼬리표(JSON) | `source, publisher, category, year, page` (+ 결과 확인용 `text`, `chunk_id`) |

> Qdrant가 꺼져 있으면 터미널에서 `docker start qdrant` (처음이면 README의 `docker run`).

In [10]:
# ─────────────────────────────────────────────────────────────
# Step 5-1. Qdrant 서버 연결 + 컬렉션(저장 공간) 만들기
#
# - size=1024        : 벡터 차원 (BGE-M3 dense)
# - Distance.COSINE  : 코사인 유사도로 비교 (FAISS의 내적과 같은 점수가 나온다)
# - 이미 같은 이름의 컬렉션이 있으면 지우고 새로 만든다
#   → 노트북을 여러 번 실행해도 데이터가 중복으로 쌓이지 않게
# ─────────────────────────────────────────────────────────────
from qdrant_client import QdrantClient
from qdrant_client import models   # PointStruct, Filter 등 Qdrant 전용 "양식"이 모여 있는 곳

client = QdrantClient(host="localhost", port=6333, timeout=60, check_compatibility=False)

COLLECTION = "policy_reports"

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
    print(f"🗑️ 기존 '{COLLECTION}' 컬렉션 삭제 (새로 만들기 위해)")

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE),
)
print(f"✅ 컬렉션 준비 완료: '{COLLECTION}' (vector size={DIM}, distance=COSINE)")

🗑️ 기존 'policy_reports' 컬렉션 삭제 (새로 만들기 위해)
✅ 컬렉션 준비 완료: 'policy_reports' (vector size=1024, distance=COSINE)


In [11]:
# ─────────────────────────────────────────────────────────────
# Step 5-2. 청크 2,197개 업로드 (id + vector + payload)
#
# PointStruct = Qdrant에 넣을 "한 줄"
#   id      → chunk_id 그대로 사용 (★ 공통 열쇠: chunks[i] ↔ embeddings[i] ↔ Qdrant id i)
#   vector  → embeddings[i] (numpy 배열은 .tolist()로 일반 리스트로 바꿔서 전달)
#   payload → 필터에 쓸 꼬리표 + 결과 확인용 원문
#
# upload_points 는 내부에서 batch_size개씩 나눠서 보낸다.
# wait=True → 서버가 "저장 끝" 할 때까지 기다린다 (바로 다음 셀에서 개수를 세기 위해)
# ─────────────────────────────────────────────────────────────
PAYLOAD_KEYS = ["source", "publisher", "category", "year", "page"]

points = [
    models.PointStruct(
        id=c["chunk_id"],
        vector=embeddings[c["chunk_id"]].tolist(),
        payload={
            **{k: c[k] for k in PAYLOAD_KEYS},   # 필터용 꼬리표 5개
            "chunk_id": c["chunk_id"],
            "text": c["text"],                    # 검색 결과를 바로 읽기 위해 원문도 함께 저장
        },
    )
    for c in chunks
]

start = time.time()
client.upload_points(collection_name=COLLECTION, points=points, batch_size=256, wait=True)
print(f"⏱️ 업로드: {time.time() - start:.1f}초")

n = client.count(COLLECTION, exact=True).count
print(f"✅ 업로드 완료: Qdrant에 {n}개 저장 (청크 {len(chunks)}개와 같아야 함)")

⏱️ 업로드: 0.5초
✅ 업로드 완료: Qdrant에 2197개 저장 (청크 2197개와 같아야 함)


In [12]:
# ─────────────────────────────────────────────────────────────
# Step 5-3. 필터에 쓸 필드에 "색인(payload index)" 만들기
#
# 책 뒤의 "찾아보기"와 같다. 색인이 없어도 필터는 되지만,
# 데이터가 많아지면 모든 포인트의 꼬리표를 하나씩 열어봐야 해서 느려진다.
#   category  → KEYWORD (정확히 일치하는 문자열: "반도체", "AI동향" ...)
#   publisher → KEYWORD
#   year      → INTEGER (숫자라서 >=, <= 같은 범위 비교 가능)
# ─────────────────────────────────────────────────────────────
for field, schema in [
    ("category", models.PayloadSchemaType.KEYWORD),
    ("publisher", models.PayloadSchemaType.KEYWORD),
    ("year", models.PayloadSchemaType.INTEGER),
]:
    client.create_payload_index(COLLECTION, field_name=field, field_schema=schema, wait=True)

info = client.get_collection(COLLECTION)
print("✅ payload 색인 준비 완료:", {k: v.data_type.value for k, v in info.payload_schema.items()})

# 저장된 포인트 하나를 꺼내서 모양 확인 (벡터는 너무 길어서 빼고)
sample = client.retrieve(COLLECTION, ids=[0], with_payload=True, with_vectors=False)[0]
print(f"\n📦 id={sample.id} 의 payload:")
print(json.dumps({k: (v[:60] + "..." if k == "text" else v) for k, v in sample.payload.items()}, ensure_ascii=False, indent=2))

✅ payload 색인 준비 완료: {'category': 'keyword', 'publisher': 'keyword', 'year': 'integer'}

📦 id=0 의 payload:
{
  "source": "1. SPRi AI Brief_10월호_산업동향_202510",
  "publisher": "SPRi",
  "category": "AI동향",
  "year": 2025,
  "page": 2,
  "chunk_id": 0,
  "text": "CONTENTS\n∙중국 국무원, ‘AI 플러스’ 심화 추진을 위한 정책 로드맵 발표\n∙대만 행정원, AI 기..."
}


In [13]:
# ─────────────────────────────────────────────────────────────
# Step 5-4. Qdrant 검색 함수
#
# query_filter=None 이면 조건 없이 전체에서 검색 (FAISS와 같은 상황)
# 결과(ScoredPoint)에는 id, score, payload 가 들어 있다.
# ─────────────────────────────────────────────────────────────
def qdrant_search(query: str, k: int = 5, query_filter: models.Filter | None = None) -> list[tuple[int, float]]:
    """Qdrant에서 검색해서 (chunk_id, 점수) 목록을 돌려준다. show_results()에 그대로 넣을 수 있다."""
    response = client.query_points(
        collection_name=COLLECTION,
        query=embed_query(query).tolist(),
        query_filter=query_filter,
        limit=k,
        with_payload=False,   # 원문은 chunks[id] 로 찾으면 되니까 굳이 받아오지 않음
    )
    return [(int(p.id), float(p.score)) for p in response.points]


print("✅ Qdrant 검색 함수 준비 완료: qdrant_search()")

✅ Qdrant 검색 함수 준비 완료: qdrant_search()


In [14]:
# ─────────────────────────────────────────────────────────────
# Step 5-5. FAISS vs Qdrant: 같은 질문이면 같은 결과가 나와야 한다
#   같은 벡터 + 같은 거리(코사인) + 같은 방식(HNSW)이므로 top-5가 일치하고 점수도 거의 같아야 정상.
# ─────────────────────────────────────────────────────────────
query = "HBM이란 무엇인가?"
f_res = faiss_search(query, k=5)
q_res = qdrant_search(query, k=5)

compare = pd.DataFrame({
    "FAISS chunk_id": [cid for cid, _ in f_res],
    "FAISS 점수": [round(s, 4) for _, s in f_res],
    "Qdrant chunk_id": [cid for cid, _ in q_res],
    "Qdrant 점수": [round(s, 4) for _, s in q_res],
}, index=range(1, 6))
print(f"🔎 질문: {query}")
print(f"top-5 일치 여부: {'✅ 같음' if [c for c, _ in f_res] == [c for c, _ in q_res] else '⚠️ 순서가 다름 (점수 차이가 아주 작으면 근사 검색 특성상 정상)'}")
compare

🔎 질문: HBM이란 무엇인가?
top-5 일치 여부: ✅ 같음


,FAISS chunk_id,FAISS 점수,Qdrant chunk_id,Qdrant 점수
1,2168,0.5848,2168,0.5846
2,1754,0.5784,1754,0.5784
3,2165,0.5733,2165,0.5732
4,2166,0.5727,2166,0.5725
5,2167,0.5571,2167,0.5568


## Step 6. 메타데이터 필터 검색

Qdrant의 진짜 장점. **"의미가 비슷한 것"** + **"조건에 맞는 것"**을 한 번에 찾는다.

| 하고 싶은 것 | Qdrant 필터 |
|---|---|
| 반도체 문서에서만 | `FieldCondition(key="category", match=MatchValue(value="반도체"))` |
| 2026년 이후 자료만 | `FieldCondition(key="year", range=Range(gte=2026))` |
| 여러 조건 모두 만족 (AND) | `Filter(must=[조건1, 조건2])` |
| 여러 조건 중 하나 (OR) | `Filter(should=[조건1, 조건2])` |
| 이 조건은 빼고 (NOT) | `Filter(must_not=[조건])` |

01의 관찰 메모에서 **SPRi + ETRI가 전체의 약 3/4**이라고 했다. 반도체 질문을 해도 AI 뉴스가 섞여 나올 수 있는데, 필터로 이 문제를 어떻게 해결하는지 본다.

In [15]:
# ─────────────────────────────────────────────────────────────
# Step 6-1. 지금 데이터에 어떤 category / year 값이 있는지 먼저 확인
#   필터 값은 "정확히 일치"해야 하므로, 오타 하나만 있어도 0건이 된다.
# ─────────────────────────────────────────────────────────────
df_chunks = pd.DataFrame(chunks)
print("📂 category별 청크 수:", df_chunks["category"].value_counts().to_dict())
print("📅 year별 청크 수    :", df_chunks["year"].value_counts().sort_index().to_dict())

# Qdrant 쪽에서도 같은 숫자가 나오는지 count + filter 로 확인
for cat in df_chunks["category"].unique():
    cnt = client.count(
        COLLECTION,
        count_filter=models.Filter(must=[models.FieldCondition(key="category", match=models.MatchValue(value=cat))]),
        exact=True,
    ).count
    print(f"   Qdrant category='{cat}': {cnt}개")

📂 category별 청크 수: {'AI동향': 1040, 'AI전략': 588, '반도체': 569}
📅 year별 청크 수    : {2020: 588, 2024: 54, 2025: 1124, 2026: 431}
   Qdrant category='AI동향': 1040개
   Qdrant category='AI전략': 588개
   Qdrant category='반도체': 569개


In [16]:
# ─────────────────────────────────────────────────────────────
# Step 6-2. category 필터: "반도체" 문서에서만 검색
#
# 같은 질문을 ① 필터 없이 ② category="반도체" 로 검색해서 비교한다.
# 질문은 일부러 반도체·AI 뉴스 양쪽에 다 나오는 "엔비디아 GPU"로 골랐다.
# → 필터가 없으면 SPRi(AI 뉴스) 기사가 상위를 차지한다.
# ─────────────────────────────────────────────────────────────
semi_filter = models.Filter(
    must=[  # must = 아래 조건을 "모두" 만족해야 함 (AND)
        models.FieldCondition(key="category", match=models.MatchValue(value="반도체")),
    ]
)

query = "엔비디아 GPU 신제품"
print(f"🔎 질문: {query}\n")
print("① 필터 없음")
print(show_results(qdrant_search(query, k=5), preview=40).to_string())
print("\n② category = '반도체'")
semi_results = qdrant_search(query, k=5, query_filter=semi_filter)
print(show_results(semi_results, preview=40).to_string())

# 필터가 제대로 걸렸는지 자동 검사: 결과가 전부 반도체 문서여야 한다
assert all(chunks[cid]["category"] == "반도체" for cid, _ in semi_results)
print("\n✅ 검증: 필터 결과 5개 모두 category='반도체'")

🔎 질문: 엔비디아 GPU 신제품

① 필터 없음
       점수         발행처    분류    연도   쪽  chunk_id                                   내용 미리보기
순위                                                                                       
1   0.617  KDB미래전략연구소   반도체  2024   7      2154  2024. 7 제824호  9 ○엔비디아의 AI GPU는 MLPerf4)
2   0.609  KDB미래전략연구소   반도체  2024   6      2151  Ⅱ. 핵심 분야별 기술 동향 1. 프로세서 : AI GPU, NPU, 뉴
3   0.583        SPRi  AI동향  2026  18       651  엔비디아 에서 생태계 확장 선언 , GTC 2026 엔비디아의 젠슨 황 
4   0.583        SPRi  AI동향  2025  14       901  대륙을 AI 산업 혁명의 선두 주자로 자리매김하기 위한 AI 인프라를 구
5   0.572        SPRi  AI동향  2025  13       896  AI 소프트웨어 플랫폼 ‘ROCm 7’을 발표하는 한편, 개발자 지원 플

② category = '반도체'
       점수         발행처   분류    연도   쪽  chunk_id                                   내용 미리보기
순위                                                                                      
1   0.617  KDB미래전략연구소  반도체  2024   7      2154  2024. 7 제824호  9 ○엔비디아의 AI GPU는 MLPerf4)
2   0.609  KDB미래전략연구소  반도체  2024   6      2151  Ⅱ. 핵심 분

In [17]:
# ─────────────────────────────────────────────────────────────
# Step 6-3. year 필터: 2026년 이후 자료만 (범위 조건)
#
# Range(gte=2026) → year >= 2026
#   gte = greater than or equal (이상), gt = 초과, lte = 이하, lt = 미만
# "최신 동향"을 물을 때 2020년 ETRI 전략 보고서가 섞이는 걸 막아준다.
# ─────────────────────────────────────────────────────────────
recent_filter = models.Filter(
    must=[models.FieldCondition(key="year", range=models.Range(gte=2026))]
)

query = "최근 생성형 AI 모델 출시 동향"
print(f"🔎 질문: {query}\n")
print("① 필터 없음")
print(show_results(qdrant_search(query, k=5), preview=40).to_string())
print("\n② year >= 2026")
recent_results = qdrant_search(query, k=5, query_filter=recent_filter)
print(show_results(recent_results, preview=40).to_string())

assert all(chunks[cid]["year"] >= 2026 for cid, _ in recent_results)
print("\n✅ 검증: 필터 결과 5개 모두 year >= 2026")

🔎 질문: 최근 생성형 AI 모델 출시 동향

① 필터 없음


       점수   발행처    분류    연도   쪽  chunk_id                                   내용 미리보기
순위                                                                                 
1   0.664  SPRi  AI동향  2025   8       112  효율적인 솔루션을 원하는 사용자 수요에 대응해 경량 모델을 적극적으로 출
2   0.638  SPRi  AI동향  2025   2       180  ▹ LG AI연구원, 엑사원 3.5 기반모델 3종 오픈소스로 공개····
3   0.621  SPRi  AI동향  2026   3       603  25.2.17 54.1 200k Anthropic GLM-5 26.2.1
4   0.616  SPRi  AI동향  2026   3       778  26.2.19. 118.7 1m Google GPT-5.4(xhigh) 
5   0.613  SPRi  AI동향  2026  21       321  ∙데이터 큐레이션과 합성 데이터 생성·평가를 포함한 단계별 설명과 예제를

② year >= 2026
       점수   발행처    분류    연도   쪽  chunk_id                                   내용 미리보기
순위                                                                                 
1   0.621  SPRi  AI동향  2026   3       603  25.2.17 54.1 200k Anthropic GLM-5 26.2.1
2   0.616  SPRi  AI동향  2026   3       778  26.2.19. 118.7 1m Google GPT-5.4(xhigh) 
3   0.613  SPRi  AI동향  2026  21       321  ∙데이터 큐레이션과 합성 데이터

In [18]:
# ─────────────────────────────────────────────────────────────
# Step 6-4. 조건 여러 개 조합 (AND + NOT)
#
# "반도체 문서 중에서, 2025년 이후 자료만, 단 삼일PwC는 제외"
#   must     : 반도체 AND year >= 2025
#   must_not : publisher = 삼일PwC 는 빼기
# (01에서 봤듯 삼일PwC는 띄어쓰기가 사라진 텍스트라, 이런 식으로 빼고 비교해 볼 수 있다)
# ─────────────────────────────────────────────────────────────
combo_filter = models.Filter(
    must=[
        models.FieldCondition(key="category", match=models.MatchValue(value="반도체")),
        models.FieldCondition(key="year", range=models.Range(gte=2025)),
    ],
    must_not=[
        models.FieldCondition(key="publisher", match=models.MatchValue(value="삼일PwC")),
    ],
)

query = "HBM 기술 경쟁력 확보 방안"
print(f"🔎 질문: {query}")
print("   조건: category='반도체' AND year>=2025 AND publisher!='삼일PwC'\n")
combo_results = qdrant_search(query, k=5, query_filter=combo_filter)
print(show_results(combo_results, preview=45).to_string())

ok = all(
    chunks[cid]["category"] == "반도체" and chunks[cid]["year"] >= 2025 and chunks[cid]["publisher"] != "삼일PwC"
    for cid, _ in combo_results
)
assert ok
print("\n✅ 검증: 결과 모두 세 조건을 만족")

🔎 질문: HBM 기술 경쟁력 확보 방안
   조건: category='반도체' AND year>=2025 AND publisher!='삼일PwC'



       점수       발행처   분류    연도   쪽  chunk_id                                        내용 미리보기
순위                                                                                         
1   0.627  반도체특별위원회  반도체  2025  13      1852  시스템 반도체와 팹리스 경쟁력 강화를 위한 종합적 지원이 병행될 경우, 자동차, 
2   0.564  반도체특별위원회  반도체  2025  16      1859  Rebellions는 ATOM을 통해 데이터센터용 AI 가속기 시장 진입을 시도하
3   0.554  반도체특별위원회  반도체  2025  13      1853  SoC 자체 개발 역량을 단계적으로 확보할 필요가 있다. 5. 종합 요약 AI 반
4   0.552  반도체특별위원회  반도체  2025  12      1849  처리나 빠른 기능 변경이 요구되는 니치 마켓에서 지속적인 역할을 수행할 것으로 보
5   0.550  반도체특별위원회  반도체  2025  24      1884  파운드리와 결합해 사실상 풀스택 공급 구조를 강화하고 있다. 한국은 하나마이크론,

✅ 검증: 결과 모두 세 조건을 만족


In [19]:
# ─────────────────────────────────────────────────────────────
# Step 6-5. 왜 "검색 후 거르기"가 아니라 "검색하면서 거르기"인가?
#
# FAISS 방식(먼저 top-5 검색 → 파이썬으로 반도체만 남기기)과
# Qdrant 방식(필터를 걸고 top-5 검색)을 같은 질문으로 비교한다.
# AI 뉴스 비중이 큰 질문일수록 FAISS 방식은 남는 결과가 줄어든다.
# ─────────────────────────────────────────────────────────────
rows = []
for q in ["AI 칩 수요 증가가 메모리 산업에 미치는 영향", "AI 반도체 스타트업 투자", "엔비디아 GPU 신제품"]:
    post = [cid for cid, _ in faiss_search(q, k=5) if chunks[cid]["category"] == "반도체"]
    pre = qdrant_search(q, k=5, query_filter=semi_filter)
    rows.append({"질문": q, "FAISS top5 → 반도체만 남김": f"{len(post)}개", "Qdrant 필터 top5": f"{len(pre)}개"})

pd.DataFrame(rows).set_index("질문")

,FAISS top5 → 반도체만 남김,Qdrant 필터 top5
질문,,
AI 칩 수요 증가가 메모리 산업에 미치는 영향,5개,5개
AI 반도체 스타트업 투자,5개,5개
엔비디아 GPU 신제품,2개,5개


### 📝 Step 5~6 관찰 메모
- FAISS와 Qdrant의 top-5가 **완전히 같다**. 점수는 소수점 넷째 자리에서만 조금 다르다(Qdrant가 내부에서 벡터를 다시 정규화하고 계산 방식이 조금 달라서 생기는 차이).
- **category 필터 효과:** "엔비디아 GPU 신제품"을 필터 없이 검색하면 top-5 중 3개가 SPRi(AI 뉴스)였다. FAISS 결과를 받은 뒤 거르면 **2개만 남고**, Qdrant 필터로 검색하면 **반도체 문서 5개를 꽉 채워서** 돌려준다. → 01에서 본 "SPRi+ETRI 3/4 쏠림" 문제를 필터로 해결할 수 있다.
- **year 필터 효과:** "최근 생성형 AI 모델 출시 동향"에서 2025년 청크가 빠지고 2026년 청크만 남았다. 그런데 상위권에 **모델 이름·숫자만 나열된 벤치마크 표 청크**(SPRi 2026년 호 3쪽)가 올라왔다. 표가 줄바꿈된 글자 나열로 추출되어 문장으로서 의미가 거의 없는 청크다. → 표 처리는 이후 개선 후보.
- **조합 필터(AND + NOT)**도 결과 5개가 모두 세 조건을 만족했다. `assert`로 자동 검사해 두어서, 나중에 데이터가 바뀌어도 필터가 깨지면 바로 알 수 있다.
- **Qdrant payload 색인**(category·publisher·year)은 지금 규모에서는 속도 차이가 거의 없다. 하지만 문서가 늘어날 것에 대비해 미리 만들어 둔다.

## ✅ Step 3~6 체크포인트

- [ ] Step 3: 임베딩 shape `(2197, 1024)`, `outputs/embeddings.npy` 저장 (두 번째 실행부터 "캐시 사용")
- [ ] Step 4: FAISS HNSW 검색 결과 확인, Recall@10 측정
- [ ] Step 5: Qdrant `policy_reports` 컬렉션에 2,197개 업로드, FAISS와 top-5 일치
- [ ] Step 6: category / year / 조합 필터 검색, 결과가 조건을 모두 만족

### 다음: `03_hybrid_rerank.ipynb`
Dense(의미) 검색이 "HBM" 같은 고유명사·약어를 놓치는 사례를 찾고 → BM25(키워드) 검색 → RRF로 두 결과 합치기(Hybrid) → BGE-Reranker로 재정렬